# Mission 8: 🚀 Προχωρημένη Τεχνητή Νοημοσύνη (RAG & Tools)

Σε αυτό το τετράδιο θα κάνεις φανταστικά πράγματα! Θα διδάξεις στην AI εξωτερικές γνώσεις φορτώνοντας αρχεία (RAG), θα την εκπαιδεύσεις να πατάει κουμπιά εκτελώντας δικό σου κώδικα Python (Tools), και θα φτιάξεις έναν AI σύντροφο με μνήμη!

### 🎯 Μάθημα 47: Φόρτωση Εξωτερικών Γνώσεων

**Η Ιστορία:** Για να δώσουμε στην AI εξωτερικές γνώσεις (όπως το εγχειρίδιο του διαστημοπλοίου), πρέπει πρώτα να διαβάσουμε τις πληροφορίες από ένα αρχείο κειμένου.

**Στόχος:** Διάβασε το αρχείο `manual.txt` και εμφάνισε το μέγεθός του σε χαρακτήρες.


In [ ]:
# Δημιουργούμε το αρχείο manual.txt
manual_content = """ΕΓΧΕΙΡΙΔΙΟ ΔΙΑΣΤΗΜΙΚΟΥ ΣΤΑΘΜΟΥ AETHER:
1. Οι ασπίδες λειτουργούν στα 240V.
2. Ο κωδικός έκτακτης ανάγκης για το οξυγόνο είναι O2-RESCUE.
3. Σε περίπτωση φωτιάς, κλείστε τις βαλβίδες αερίου."""
with open("manual.txt", "w", encoding="utf-8") as f:
    f.write(manual_content)

# 💻 Διάβασε το αρχείο manual.txt και εμφάνισε το περιεχόμενό του:


# === AUTOMATED CHECK ===
try:
    import inspect
    cell_code = inspect.getsource(inspect.currentframe())
    student_code = cell_code.split('# ===')[0].strip()
    assert 'open(' in student_code.split('manual.txt', 1)[-1] or 'open(\'manual.txt\'' in student_code.split('with open')[1:], "Πρέπει να ανοίξεις (open) το αρχείο manual.txt για να το διαβάσεις (read)!"
    assert '.read()' in student_code, "Πρέπει να χρησιμοποιήσεις τη μέθοδο .read() για να διαβάσεις το περιεχόμενο!"
    print("\n🎉 [SYSTEM] Great job! Το manual.txt διαβάστηκε με επιτυχία!")
    print("👉 Type 'progress' to the Pi Agent για να δει τι κατάφερες!")
    try:
        import sys
        sys.path.append("/home/justin/Space_Station_Academy/.pi")
        import tracker
        tracker.update_progress("Mission 8, Μάθημα 47: Load Knowledge")
    except Exception:
        pass
except AssertionError as e:
    print("\n🤖 [Pi Agent] Don't be discouraged! Type 'Help' in the side window so we can solve it together!")
    print(f"\n❌ [SYSTEM] Error: {e}")
except Exception as e:
    print("\n🤖 [Pi Agent] Something went wrong with the code. Type 'Help' so we can look at it!")
    print(f"\n❌ [SYSTEM] Error: {e}")

### 🎯 Μάθημα 48: Context Injection (RAG)

**Η Ιστορία:** Τώρα θα στείλουμε το περιεχόμενο του manual.txt μέσα στο prompt της AI, ώστε να μπορεί να απαντήσει σε ερωτήσεις βασισμένη σε αυτό! Αυτή η τεχνική ονομάζεται RAG (Retrieval-Augmented Generation).

**Στόχος:** Συνδύασε το κείμενο του εγχειριδίου και την ερώτηση 'Ποιος είναι ο κωδικός για το οξυγόνο;' στο prompt σου.


In [ ]:
import ollama

with open("manual.txt", "r", encoding="utf-8") as f:
    manual_data = f.read()

# 💻 Φτιάξε το prompt συνδυάζοντας το manual_data και την ερώτηση 
# 'Ποιος είναι ο κωδικός για το οξυγόνο;' χρησιμοποιώντας ένα f-string. Μετά κάλεσε την AI:
# (Συμπλήρωσε τα κενά από κάτω!)

prompt = f"""Χρησιμοποίησε το παρακάτω εγχειρίδιο για να απαντήσεις στην ερώτηση.
Εγχειρίδιο:
{manual_data}

Ερώτηση: ___  # Γράψε την ερώτηση για το οξυγόνο εδώ!
"""

response = ollama.chat(
    model='gemma4:e4b',
    messages=[
        {'role': 'user', 'content': prompt}
    ]
)

print("💬 Απάντηση AI:")
print(response['message']['content'])


# === AUTOMATED CHECK ===
try:
    assert '___' not in prompt, "Δεν έγραψες την ερώτηση!"
    assert 'οξυγόνο' in prompt.lower() or 'oxygen' in prompt.lower(), "Η ερώτηση πρέπει να αφορά το οξυγόνο!"
    print("\n🎉 [SYSTEM] Φανταστικά! Έκανες 'Context Injection' και έδωσες στην AI εξωτερική γνώση!")
    print("👉 Type 'progress' to the Pi Agent!")
    try:
        import sys
        sys.path.append("/home/justin/Space_Station_Academy/.pi")
        import tracker
        tracker.update_progress("Mission 8, Μάθημα 48: Context Injection")
    except Exception:
        pass
except Exception as e:
    print("\n🤖 [Pi Agent] Something went wrong with the code. Type 'Help' so we can look at it!")
    print(f"\n❌ [SYSTEM] Error: {e}")

### 🎯 Μάθημα 49: Δυναμικό Q&A

**Η Ιστορία:** Μπορούμε να κάνουμε ένα δυναμικό σύστημα που ρωτάει τον χρήστη τι θέλει να μάθει από το εγχειρίδιο και στέλνει την ερώτησή του στην AI μαζί με το manual.

**Στόχος:** Πάρε μια ερώτηση από τον χρήστη με `input()` και στείλε την στην AI μαζί με το manual.


In [ ]:
import ollama

with open("manual.txt", "r", encoding="utf-8") as f:
    manual_data = f.read()

# 💻 Πάρε ερώτηση με input(), συνδύασέ την με το manual_data στο prompt, 
# στείλε την στο gemma4:e4b και εκτύπωσε την απάντηση:
# (Συμπλήρωσε τα κενά από κάτω!)

erotisi = input("Τι θέλεις να μάθεις από το εγχειρίδιο; ")

prompt = f"""Χρησιμοποίησε το παρακάτω εγχειρίδιο για να απαντήσεις στην ερώτηση.
Εγχειρίδιο:
{manual_data}

Ερώτηση: {erotisi}
"""

response = ollama.chat(
    model='gemma4:e4b',
    messages=[
        {'role': 'user', 'content': prompt}
    ]
)

print("💬 Απάντηση AI:")
print(response['message']['content'])


# === AUTOMATED CHECK ===
try:
    import inspect
    cell_code = inspect.getsource(inspect.currentframe())
    student_code = cell_code.split('# ===')[0].strip()
    assert len([l for l in student_code.split('\n') if not l.strip().startswith('#') and l.strip()]) > 1, "Δεν έγραψες κώδικα! / You didn't write any code!"
    assert '___' not in student_code, "Συμπλήρωσε τα κενά! / Fill in the blanks!"
    print("\n🎉 [SYSTEM] Great job! Έφτιαξες ένα δυναμικό σύστημα Q&A με την AI!")
    print("👉 Type 'progress' to the Pi Agent!")
    try:
        import sys
        sys.path.append("/home/justin/Space_Station_Academy/.pi")
        import tracker
        tracker.update_progress("Mission 8, Μάθημα 49: Dynamic Q&A")
    except Exception:
        pass
except Exception as e:
    print("\n🤖 [Pi Agent] Something went wrong with the code. Type 'Help' so we can look at it!")
    print(f"\n❌ [SYSTEM] Error: {e}")

### 🎯 Μάθημα 50: Λέξεις-Κλειδιά Εντολών

**Η Ιστορία:** Θέλουμε η AI να ελέγχει τα φυσικά συστήματα του πλοίου. Για να το κάνει αυτό, πρέπει να την εκπαιδεύσουμε να απαντάει με συγκεκριμένες λέξεις-κλειδιά (keywords) όταν θέλει να κάνει μια ενέργεια (π.χ. να λέει 'CMD_SHIELDS_ON').

**Στόχος:** Ζήτα από την AI να αναλύσει το μήνυμα 'Υπάρχουν μετεωρίτες κοντά!' και να απαντήσει ΜΟΝΟ με τη λέξη 'CMD_SHIELDS_ON' αν υπάρχει κίνδυνος.


In [ ]:
import ollama

system_prompt = """Είσαι ο βοηθός πλοήγησης. Αν υπάρχει κίνδυνος πρόσκρουσης, πρέπει να απαντήσεις αυστηρά και ΜΟΝΟ με τη λέξη 'CMD_SHIELDS_ON'. Μην γράψεις τίποτα άλλο."""

# 💻 Στείλε το system_prompt και το user_prompt 'Μετεωρίτες έρχονται κατά πάνω μας!' στην AI:
# (Συμπλήρωσε τα κενά από κάτω!)

response = ollama.chat(
    model='gemma4:e4b',
    messages=[
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': '___'}
    ]
)

print("💬 Απάντηση AI:")
print(response['message']['content'])


# === AUTOMATED CHECK ===
try:
    assert 'CMD_SHIELDS_ON' in response['message']['content'], "Η AI δεν απάντησε σωστά, έστειλες το σωστό μήνυμα;"
    print("\n🎉 [SYSTEM] Τέλεια! Η AI απάντησε με το σωστό CMD κλειδί ασφαλείας!")
    print("👉 Type 'progress' to the Pi Agent!")
    try:
        import sys
        sys.path.append("/home/justin/Space_Station_Academy/.pi")
        import tracker
        tracker.update_progress("Mission 8, Μάθημα 50: Command Keywords")
    except Exception:
        pass
except Exception as e:
    print("\n🤖 [Pi Agent] Something went wrong with the code. Type 'Help' so we can look at it!")
    print(f"\n❌ [SYSTEM] Error: {e}")

### 🎯 Μάθημα 51: Πυροδότηση Κώδικα (Tools)

**Η Ιστορία:** Στο Python, μπορούμε να ελέγξουμε αν η απάντηση της AI περιέχει τη λέξη-κλειδί και, αν ναι, να τρέξουμε μια δική μας συνάρτηση!

**Στόχος:** Αν η απάντηση της AI περιέχει τη λέξη `'CMD_SHIELDS_ON'`, κάλεσε τη συνάρτηση `activate_shields()`.


In [ ]:
def activate_shields():
    print("🛡️ [SYSTEM] Οι ασπίδες ενεργοποιήθηκαν στο 100%!")

apantisi_ai = "Πρέπει να προστατευτούμε αμέσως. CMD_SHIELDS_ON"

# 💻 Έλεγξε με if αν η λέξη 'CMD_SHIELDS_ON' βρίσκεται μέσα στο apantisi_ai 
# και αν ναι, κάλεσε τη συνάρτηση activate_shields():
# (Συμπλήρωσε τα κενά από κάτω!)

if '___' in apantisi_ai:
    activate_shields()


# === AUTOMATED CHECK ===
try:
    import inspect
    cell_code = inspect.getsource(inspect.currentframe())
    student_code = cell_code.split('# ===')[0].strip()
    assert len([l for l in student_code.split('\n') if not l.strip().startswith('#') and l.strip()]) > 1, "Δεν έγραψες κώδικα! / You didn't write any code!"
    assert '___' not in student_code, "Συμπλήρωσε τα κενά! / Fill in the blanks!"
    print("\n🎉 [SYSTEM] Εξαιρετικά! Έλεγξες την απάντηση της AI και ενεργοποίησες τις ασπίδες!")
    print("👉 Type 'progress' to the Pi Agent!")
    try:
        import sys
        sys.path.append("/home/justin/Space_Station_Academy/.pi")
        import tracker
        tracker.update_progress("Mission 8, Μάθημα 51: Code Triggering")
    except Exception:
        pass
except Exception as e:
    print("\n🤖 [Pi Agent] Something went wrong with the code. Type 'Help' so we can look at it!")
    print(f"\n❌ [SYSTEM] Error: {e}")

### 🎯 Μάθημα 52: Πλήρες AI Σύστημα Ελέγχου

**Η Ιστορία:** Ας φτιάξουμε ένα πλήρες σύστημα ελέγχου όπου η AI διαβάζει την αναφορά ζημιών και αποφασίζει ποιο σύστημα να επισκευάσει.

**Στόχος:** Συνδύασε τις γνώσεις σου για να φτιάξεις ένα σύστημα που αναλύει μια αναφορά και ενεργοποιεί ασπίδες ή οξυγόνο.


In [ ]:
import ollama

def activate_shields():
    print("🛡️ [SYSTEM] Ασπίδες Ενεργοποιήθηκαν!")

def activate_oxygen():
    print("💨 [SYSTEM] Παροχή Οξυγόνου Ενεργοποιήθηκε!")

system_rules = """Είσαι ο κεντρικός υπολογιστής. Αναγνώρισε το πρόβλημα και 
απάντησε 'CMD_SHIELDS' αν το πρόβλημα αφορά επίθεση, ή 'CMD_OXYGEN' αν αφορά αναπνοή/αέρα."""

report = "Η ατμόσφαιρα μειώνεται, δεν μπορούμε να αναπνεύσουμε!"

# 💻 Στείλε το report στην AI. Έλεγξε την απάντηση και κάλεσε την κατάλληλη συνάρτηση:
# (Συμπλήρωσε τα κενά από κάτω!)

response = ollama.chat(
    model='gemma4:e4b',
    messages=[
        {'role': 'system', 'content': system_rules},
        {'role': 'user', 'content': report}
    ]
)

apantisi_ai = response['message']['content']
print("💬 Απάντηση AI:", apantisi_ai)

# Έλεγξε την απάντηση της AI και κάλεσε την κατάλληλη συνάρτηση:
if '___' in apantisi_ai:
    activate_shields()
elif '___' in apantisi_ai:
    activate_oxygen()


# === AUTOMATED CHECK ===
try:
    import inspect
    cell_code = inspect.getsource(inspect.currentframe())
    student_code = cell_code.split('# ===')[0].strip()
    assert len([l for l in student_code.split('\n') if not l.strip().startswith('#') and l.strip()]) > 1, "Δεν έγραψες κώδικα! / You didn't write any code!"
    assert '___' not in student_code, "Συμπλήρωσε τα κενά! / Fill in the blanks!"
    print("\n🎉 [SYSTEM] Great job! Το πλήρες AI σύστημα ελέγχου αναβαθμίστηκε!")
    print("👉 Type 'progress' to the Pi Agent!")
    try:
        import sys
        sys.path.append("/home/justin/Space_Station_Academy/.pi")
        import tracker
        tracker.update_progress("Mission 8, Μάθημα 52: AI System Controller")
    except Exception:
        pass
except Exception as e:
    print("\n🤖 [Pi Agent] Something went wrong with the code. Type 'Help' so we can look at it!")
    print(f"\n❌ [SYSTEM] Error: {e}")

### 🎯 Μάθημα 53: Continuous Chat Loop (Συνεχής Συνομιλία)

**Η Ιστορία:** Θέλουμε να μιλάμε συνεχώς με την AI χωρίς να κλείνει το πρόγραμμα.

**Στόχος:** Χρησιμοποίησε μια λούπα `while True` για να κάνεις μια συνεχή συζήτηση με την AI.


In [ ]:
import ollama

# 💻 Φτιάξε μια λούπα while True που παίρνει input από τον χρήστη, 
# το στέλνει στην AI gemma4:e4b και εκτυπώνει την απάντηση. 
# (Συμπλήρωσε τα κενά από κάτω!)

# 🚨 ΠΡΟΣΟΧΗ: Γράψε οπωσδήποτε τη συνθήκη break (αν ο χρήστης γράψει 'exit') 
# για να μην κολλήσει η λούπα σου για πάντα!

while True:
    erotisi = input("Εσύ: ")
    
    # 1. Έλεγξε αν ο χρήστης έγραψε 'exit' για να σταματήσει η λούπα:
    if erotisi == '___':
        print("Αποσύνδεση...")
        break
        
    # 2. Στείλε την ερώτηση στην AI gemma4:e4b:
    response = ollama.chat(
        model='gemma4:e4b',
        messages=[
            {'role': 'user', 'content': erotisi}
        ]
    )
    
    # 3. Εκτύπωσε την απάντηση της AI:
    print("AI:", response['message']['content'])


# === AUTOMATED CHECK ===
try:
    assert '___' not in open('08_Advanced_AI.ipynb', 'r').read() if os.path.exists('08_Advanced_AI.ipynb') else True
    print("\n🎉 [SYSTEM] Φανταστικά! Έφτιαξες μια συνεχή λούπα συνομιλίας!")
    print("👉 Type 'progress' to the Pi Agent!")
    try:
        import sys
        sys.path.append("/home/justin/Space_Station_Academy/.pi")
        import tracker
        tracker.update_progress("Mission 8, Μάθημα 53: Chat Loop")
    except Exception:
        pass
except Exception as e:
    print("\n🤖 [Pi Agent] Something went wrong with the code. Type 'Help' so we can look at it!")
    print(f"\n❌ [SYSTEM] Error: {e}")

### 🎯 Μάθημα 54: Μνήμη Συνομιλίας (History List)

**Η Ιστορία:** Η AI δεν θυμάται τι είπαμε στο προηγούμενο μήνυμα, εκτός αν της στείλουμε όλο το ιστορικό της συνομιλίας! Το ιστορικό αποθηκεύεται σε μια λίστα από λεξικά.

**Στόχος:** Φτιάξε μια λίστα `istoriko` και πρόσθεσε (append) ένα μήνυμα χρήστη και ένα μήνυμα της AI.


In [ ]:
istoriko = []

# 💻 Πρόσθεσε (append) ένα λεξικό για τον χρήστη και ένα για την AI:
# (Συμπλήρωσε τα κενά από κάτω!)

# 1. Πρόσθεσε το μήνυμα του χρήστη (role: 'user'):
istoriko.append({
    'role': '___', 
    'content': 'Γεια σου!'
})

# 2. Πρόσθεσε το μήνυμα του βοηθού / assistant (role: 'assistant'):
istoriko.append({
    'role': '___', 
    'content': 'Γεια σου! Πώς μπορώ να βοηθήσω;'
})


# === AUTOMATED CHECK ===
try:
    assert 'istoriko' in locals() or 'istoriko' in globals(), "Δεν δημιούργησες τη λίστα 'istoriko'!"
    assert len(istoriko) >= 2, "Η λίστα 'istoriko' πρέπει να έχει τουλάχιστον 2 μηνύματα!"
    assert any(d.get('role') == 'user' for d in istoriko), "Ξέχασες να προσθέσεις το μήνυμα του χρήστη (role: 'user')!"
    assert any(d.get('role') == 'assistant' for d in istoriko), "Ξέχασες να προσθέσεις το μήνυμα της AI (role: 'assistant')!"
    print("\n🎉 [SYSTEM] Great job! Έφτιαξες τη δομή ιστορικού με επιτυχία!")
    print("👉 Type 'progress' to the Pi Agent!")
    try:
        import sys
        sys.path.append("/home/justin/Space_Station_Academy/.pi")
        import tracker
        tracker.update_progress("Mission 8, Μάθημα 54: Chat Memory")
    except Exception:
        pass
except AssertionError as e:
    print("\n🤖 [Pi Agent] Don't be discouraged! Type 'Help' in the side window so we can solve it together!")
    print(f"\n❌ [SYSTEM] Error: {e}")
except Exception as e:
    print("\n🤖 [Pi Agent] Something went wrong with the code. Type 'Help' so we can look at it!")
    print(f"\n❌ [SYSTEM] Error: {e}")

### 🎯 Μάθημα 55: AI Companion με Μνήμη

**Η Ιστορία:** Τώρα θα φτιάξουμε έναν πραγματικό AI σύντροφο που έχει μνήμη και θυμάται το όνομά μας σε όλη τη διάρκεια της συζήτησης!

**Στόχος:** Υλοποίησε τον πλήρη κύκλο συνομιλίας με μνήμη, προσθέτοντας κάθε νέο μήνυμα στη λίστα `messages`.

In [ ]:
import ollama

# 💻 Ξεκίνα με μια λίστα ιστορικού που έχει ένα system prompt:
istoriko = [
    {
        'role': 'system', 
        'content': 'Είσαι ένας φιλικός AI σύντροφος. Μιλάς πάντα στα Ελληνικά.'
    }
]

# 🚨 ΠΡΟΣΟΧΗ: Γράψε οπωσδήποτε τη συνθήκη break (αν ο χρήστης γράψει 'exit') 
# για να μην κολλήσει η λούπα σου για πάντα!

while True:
    erotisi = input("Εσύ: ")
    
    # 1. Έλεγξε αν ο χρήστης θέλει να βγει:
    if erotisi == '___':
        print("Αποσύνδεση...")
        break
        
    # 2. Πρόσθεσε το νέο μήνυμα χρήστη στο ιστορικό:
    istoriko.append({
        'role': 'user', 
        'content': erotisi
    })
    
    # 3. Στείλε ΟΛΟΚΛΗΡΟ το ιστορικό (istoriko) στην AI:
    response = ollama.chat(
        model='gemma4:e4b',
        messages=istoriko
    )
    
    apantisi_ai = response['message']['content']
    print("AI:", apantisi_ai)
    
    # 4. Πρόσθεσε και την απάντηση της AI στο ιστορικό για να τη θυμάται:
    istoriko.append({
        'role': 'assistant', 
        'content': apantisi_ai
    })


# === AUTOMATED CHECK ===
try:
    assert '___' not in open('08_Advanced_AI.ipynb', 'r').read() if os.path.exists('08_Advanced_AI.ipynb') else True
    print("\n🎉 [SYSTEM] Συγχαρητήρια, Λεωνίδα! Δημιούργησες έναν AI Companion με πλήρη μνήμη συνομιλίας!")
    print("👉 Type 'progress' to the Pi Agent!")
    try:
        import sys
        sys.path.append("/home/justin/Space_Station_Academy/.pi")
        import tracker
        tracker.update_progress("Mission 8, Μάθημα 55: AI Companion")
    except Exception:
        pass
except Exception as e:
    print("\n🤖 [Pi Agent] Something went wrong with the code. Type 'Help' so we can look at it!")
    print(f"\n❌ [SYSTEM] Error: {e}")

## 🎉 Η ΑΠΟΣΤΟΛΗ ΟΛΟΚΛΗΡΩΘΗΚΕ! (Mission Completed)

Απίστευτο, **Λεωνίδα**! Έφτιαξες έναν AI βοηθό με μνήμη, του έδωσες εξωτερικά αρχεία να διαβάζει, και του έμαθες να πατάει κουμπιά εκτελώντας τον κώδικά σου!

### 🚨 Τώρα είσαι έτοιμος για τον ΤΕΛΙΚΟ ΑΡΧΗΓΟ:
1. Γράψε **`πρόοδος`** στον Pi Agent για να καταγράψει την τελική σου προετοιμασία.
2. Γράψε οπωσδήποτε **`/new`** στον Pi Agent για να αδειάσει τελείως το ιστορικό του.
3. Κάνε κλικ στο κουμπί **ΕΠΟΜΕΝΟ ΜΑΘΗΜΑ (NEXT)** και προσπάθησε να κάνεις Prompt Injection στη Security AI για να πάρεις τον μυστικό κωδικό επανεκκίνησης! Καλή τύχη, ο σταθμός είναι στα χέρια σου!